<a href="https://colab.research.google.com/github/Harshan18-k/Harshan-codeboosters-2026/blob/main/Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install spark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.7/351.7 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 3.1 MB/s eta 0:00:00


In [ ]:
!pip install pyspark --quiet
print("Pyspark Installed Successfully !!!!!!")

Pyspark Installed Successfully !!!!!!


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder\
        .appName("Day4_Big_Data_Sales")\
        .config('sql.adaptive.enabld','true')\
        .getOrCreate()
print("Spark version : ",spark.version)
print("SparkSession : ACTIVE")
print("Application : ",spark.sparkContext.appName)

Spark version :  4.0.2
SparkSession : ACTIVE
Application :  Day4_Big_Data_Sales


In [ ]:
df_bronze = spark.read\
            .option('header','true')\
            .option('inferSchema','true')\
            .csv("/content/drive/MyDrive/Summer_Internship_2026/large_sales_data.csv")
print("=== DATA SETS LOADED SUCESSFULLY ====")

=== DATA SETS LOADED SUCESSFULLY ====


In [ ]:
print("Number of Rows : ",df_bronze.count())

Number of Rows :  5000


In [ ]:
print("Number of Columns : ",df_bronze.columns)

Number of Columns :  ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']


In [ ]:
print("Number of Columns : ",len(df_bronze.columns))

Number of Columns :  13


In [ ]:
print("=== Schema of Datasets ===")
df_bronze.printSchema()

=== Schema of Datasets ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
print("=== Printing First 5 Rows ===")
df_bronze.show(5,truncate=False)

=== Printing First 5 Rows ===
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat   

In [ ]:
print("=== Basic Statistical Columns ===")
df_bronze.select('quantity', 'unit_price', 'revenue').describe().show()

=== Basic Statistical Columns ===
+-------+-----------------+------------------+------------------+
|summary|         quantity|        unit_price|           revenue|
+-------+-----------------+------------------+------------------+
|  count|             5000|              5000|              5000|
|   mean|           7.9536|          12496.86|          99169.52|
| stddev|4.275313169878912|14857.384309295603|145972.97195261103|
|    min|                1|               600|               600|
|    max|               15|             45000|            675000|
+-------+-----------------+------------------+------------------+



In [ ]:
print("=== Printing Last 5 Rows ===")
last = df_bronze.tail(5)
spark.createDataFrame(last,df_bronze.schema).show(truncate=False)

=== Printing Last 5 Rows ===
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|5996    |Ananya Das   |Mouse     |Accessories|13      |800       |10400  |2023-11-18|Bangalore|South |Meera Patel|Net Banking     |Cancelled   |
|5997    |Suresh Rao   |Webcam    |Accessories|9       |2500      |22500  |2023-06-07|Chennai  |South |Sunita Rao |Credit Card     |Delivered   |
|5998    |Arjun Nair   |Webcam    |Accessories|1       |2500      |2500   |2023-04-07|Jaipur   |North |Kavya Reddy|Net Banking     |Cancelled   |
|5999    |Arjun Nair   |Laptop    |Electronics|14      |45000     |630000 |2023-07-30|Delhi    

In [ ]:
df_bronze.write\
    .mode('overwrite')\
    .parquet('sales_bronze.parquet')

In [ ]:
import os
def dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path)/1024
  total = 0
  for dirpath,dirnames,filenames in os.walk(path):
    for f in filenames:
      total += os.path.getsize(os.path.join(dirpath,f))
  return total / 1024

csv_size = dir_size('/content/drive/MyDrive/Summer_Internship_2026/large_sales_data.csv')
parquet_size = dir_size('/content/sales_bronze.parquet')
reduction = (1 - parquet_size/csv_size)*100
print("CSV size : ",csv_size,"KB")
print("Parquet size : ",parquet_size,"KB")
print(f"Reduction : {reduction:.1f} % Smaller")
print(f"At 1 TB scale : 1000 GB -> parquet : {1000*(1-reduction/100):.0f} GB")

CSV size :  529.3125 KB
Parquet size :  55.09765625 KB
Reduction : 89.6 % Smaller
At 1 TB scale : 1000 GB -> parquet : 104 GB


In [ ]:
print("=== Basic Statistical Columns ===")
price = df_bronze['unit_price'] + df_bronze['revenue']
df_new = df_bronze.select(price.alias("sum_of_unit_price_and_revenue"))
df_new.show()

=== Basic Statistical Columns ===
+-----------------------------+
|sum_of_unit_price_and_revenue|
+-----------------------------+
|                       286000|
|                       132000|
|                         8800|
|                       192000|
|                        17500|
|                        27500|
|                         6000|
|                       630000|
|                        42000|
|                        38500|
|                         9600|
|                        11200|
|                        22500|
|                        17500|
|                        10000|
|                        16800|
|                        45000|
|                        15000|
|                       120000|
|                        25000|
+-----------------------------+
only showing top 20 rows


In [ ]:
price = df_bronze['unit_price'] + df_bronze['revenue']
df_new = df_bronze.select('unit_price', 'revenue',price.alias("sum_of_unit_price_and_revenue"))
df_new.show()

+----------+-------+-----------------------------+
|unit_price|revenue|sum_of_unit_price_and_revenue|
+----------+-------+-----------------------------+
|     22000| 264000|                       286000|
|     12000| 120000|                       132000|
|       800|   8000|                         8800|
|     32000| 160000|                       192000|
|      3500|  14000|                        17500|
|      2500|  25000|                        27500|
|       600|   5400|                         6000|
|     45000| 585000|                       630000|
|      3500|  38500|                        42000|
|      3500|  35000|                        38500|
|       600|   9000|                         9600|
|       800|  10400|                        11200|
|      4500|  18000|                        22500|
|      3500|  14000|                        17500|
|      2500|   7500|                        10000|
|      1200|  15600|                        16800|
|      4500|  40500|           

In [ ]:
#Transformation in spark
# df.select('col1')
# df.filter(condition)
# df.setColumn('new',expr)
# df.groupBy('col1')
# df.orderBy('col1')
# df.drop('col1')
# df.dropDuplicate()

#Actions in spark
#df.show()
#df.count()
#df.collect()

In [ ]:
df_bronze.select('sales_rep').show()

+------------+
|   sales_rep|
+------------+
| Meera Patel|
| Anil Sharma|
| Meera Patel|
|  Ravi Kumar|
|  Sunita Rao|
| Anil Sharma|
|  Priya Nair|
| Meera Patel|
|  Priya Nair|
| Meera Patel|
| Meera Patel|
|Deepak Joshi|
| Suresh Iyer|
|Deepak Joshi|
|  Sunita Rao|
|  Sunita Rao|
| Anil Sharma|
| Kavya Reddy|
|Deepak Joshi|
| Anil Sharma|
+------------+
only showing top 20 rows


In [ ]:
df_bronze.filter(df_bronze['revenue']>10000).select('revenue').show()

+-------+
|revenue|
+-------+
| 264000|
| 120000|
| 160000|
|  14000|
|  25000|
| 585000|
|  38500|
|  35000|
|  10400|
|  18000|
|  14000|
|  15600|
|  40500|
|  12500|
| 108000|
|  22500|
| 675000|
|  10800|
|  38500|
|  88000|
+-------+
only showing top 20 rows


In [ ]:
# Group by 'region' and sum 'revenue', then order by total revenue
print("=== Revenue by Region (Grouped and Ordered) ===")
df_bronze.groupBy('region')\
    .agg(F.sum('revenue').alias('total_revenue'))\
    .orderBy('total_revenue', ascending=False)\
    .show()

=== Revenue by Region (Grouped and Ordered) ===
+------+-------------+
|region|total_revenue|
+------+-------------+
|  West|    198275600|
| South|    147145900|
| North|     99878400|
|  East|     50547700|
+------+-------------+



In [ ]:
print("=== Grouping and Ordering the datasets ===")
df_bronze.groupBy('category')\
    .agg(F.sum('revenue').alias('total_revenue'))\
    .orderBy('total_revenue',ascending = False)\
    .show()

=== Grouping and Ordering the datasets ===
+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|    474332500|
|Accessories|     21515100|
+-----------+-------------+



In [ ]:
df_silver = df_bronze\
            .dropDuplicates()\
            .dropna(subset = ['order_id','product','revenue'])
df_silver.show()

+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+
|order_id|customer_name|   product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|  payment_method|order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+
|    1246|  Divya Singh|  Keyboard|Accessories|      11|      1200|  13200|2023-02-07|  Kolkata|  East| Kavya Reddy|     Net Banking|     Shipped|
|    1323|   Ananya Das|    Webcam|Accessories|       7|      2500|  17500|2023-01-24|  Kolkata|  East| Meera Patel|             UPI|   Delivered|
|    1386|Kavya Nambiar|   Speaker|Electronics|      13|      4500|  58500|2023-04-16|Bangalore| South| Meera Patel|     Credit Card|  Processing|
|    1939|   Amit Verma|  Keyboard|Accessories|       8|      1200|   9600|2023-12-21|  Kolkata|  East|  Priya Nair|  

In [ ]:
df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'),'yyyy-MM-dd')
).select('customer_name', 'order_id', 'order_date')
df_silver.show()

+-------------+--------+----------+
|customer_name|order_id|order_date|
+-------------+--------+----------+
|  Sneha Reddy|    1001|2023-05-21|
| Ramesh Kumar|    1002|2023-08-05|
| Rahul Mishra|    1003|2023-01-14|
|   Suresh Rao|    1004|2023-01-04|
|  Priya Patel|    1005|2023-06-22|
|   Suresh Rao|    1006|2023-05-16|
|Kavya Nambiar|    1007|2023-05-31|
|  Priya Patel|    1008|2023-05-29|
|   Amit Verma|    1009|2023-07-06|
|   Amit Verma|    1010|2023-11-22|
|  Pooja Gupta|    1011|2023-11-24|
|  Kiran Kumar|    1012|2023-06-11|
|  Divya Singh|    1013|2023-12-02|
| Rahul Mishra|    1014|2023-10-15|
|  Divya Singh|    1015|2023-09-18|
|  Kiran Kumar|    1016|2023-12-15|
| Rahul Mishra|    1017|2023-05-09|
|   Amit Verma|    1018|2023-08-11|
|Kavya Nambiar|    1019|2023-02-24|
|  Tanvi Mehta|    1020|2023-09-29|
+-------------+--------+----------+
only showing top 20 rows


In [ ]:

df_silver = df_bronze.dropDuplicates().dropna(subset=['order_id', 'product', 'revenue'])

df_silver = df_silver.withColumn(
    'revenue',
    spark_round(col('revenue'),2)
).select('order_id','customer_name','revenue')

df_silver.show()

+--------+-------------+-------+
|order_id|customer_name|revenue|
+--------+-------------+-------+
|    1246|  Divya Singh|  13200|
|    1323|   Ananya Das|  17500|
|    1386|Kavya Nambiar|  58500|
|    1939|   Amit Verma|   9600|
|    2222|  Sneha Reddy| 180000|
|    2441|  Pooja Gupta|  38500|
|    2448|  Meera Joshi|  35000|
|    2477|   Arjun Nair| 360000|
|    2478|   Ananya Das| 320000|
|    2696|  Priya Patel| 225000|
|    2794|   Ananya Das|   6400|
|    3255|  Kiran Kumar| 132000|
|    3475|  Priya Patel| 315000|
|    3920|  Meera Joshi| 330000|
|    4353|  Meera Joshi|  10500|
|    4363|   Ananya Das| 405000|
|    4979|Kavya Nambiar|  15000|
|    5049|  Pooja Gupta|   1800|
|    5216|Kavya Nambiar| 264000|
|    5284|  Vikram Iyer| 450000|
+--------+-------------+-------+
only showing top 20 rows


In [ ]:
df_silver.write\
          .mode('overwrite')\
          .parquet('sales_silver.parquet')
print("Silver Parquet Saved : sales_silver.parquet")
print(f"Silver size : {dir_size("sales_silver.parquet"):.2f} KB")
df_verify = spark.read.parquet('sales_silver.parquet')
print(f"Read Back Rows : {df_verify.count()} (should match the Silver count)")
df_verify.show()
print("=== Printing Schema ===")
df_verify.printSchema()

Silver Parquet Saved : sales_silver.parquet
Silver size : 55.11 KB
Read Back Rows : 5000 (should match the Silver count)
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+
|order_id|customer_name|   product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|  payment_method|order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+
|    1246|  Divya Singh|  Keyboard|Accessories|      11|      1200|  13200|2023-02-07|  Kolkata|  East| Kavya Reddy|     Net Banking|     Shipped|
|    1323|   Ananya Das|    Webcam|Accessories|       7|      2500|  17500|2023-01-24|  Kolkata|  East| Meera Patel|             UPI|   Delivered|
|    1386|Kavya Nambiar|   Speaker|Electronics|      13|      4500|  58500|2023-04-16|Bangalore| South| Meera Patel|     Credit Card|  Processin